In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cftime
import os

# Import modules
%reload_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [2]:
# from src.netcdf import mat_to_xarray
import sys
sys.path.append('/Users/yugao/UOP/ORS-processing/src')
from util import convert_cftime_to_pandas_timestamp,  truncate_valid_hourly_data

In [3]:
case_name = 'stratus14'

In [4]:
ll /Users/yugao/UOP/ORS-processing/data/processed/stratus/

total 63728
-rw-r--r--  1 yugao  staff  1286787 Apr 10 18:52 stratus12_sbe16_1876.nc
-rw-r--r--  1 yugao  staff  1286787 Apr 11 00:25 stratus12_sbe16_1879.nc
-rw-r--r--  1 yugao  staff   822282 Apr 11 01:35 stratus13_sbe16_1873.nc
-rw-r--r--  1 yugao  staff   822242 Apr 11 01:37 stratus13_sbe16_1875.nc
-rw-r--r--  1 yugao  staff  4059928 Apr 11 01:21 stratus14_sbe16_10600.nc
-rw-r--r--  1 yugao  staff  4042328 Apr 11 01:36 stratus14_sbe16_10601.nc
-rw-r--r--  1 yugao  staff  4664462 Apr 23 22:30 stratus15_sbe37_11394.nc
-rw-r--r--  1 yugao  staff  3888841 Apr 23 22:28 stratus15_sbe37_12257.nc
-rw-r--r--  1 yugao  staff  4990574 Jun 12 18:51 stratus16_sbe37_10600.nc
-rw-r--r--  1 yugao  staff  4990286 Jun 12 18:50 stratus16_sbe37_10601.nc


In [5]:
# netcdf_path = f'/Users/yugao/UOP/ORS-processing/data/processed/stratus/{case_name}_sbe16_1873.nc'
# netcdf_path2 = f'/Users/yugao/UOP/ORS-processing/data/processed/stratus/{case_name}_sbe16_1875.nc'
netcdf_path = f'/Users/yugao/UOP/ORS-processing/data/processed/stratus/{case_name}_sbe16_10601.nc'
netcdf_path2 = f'/Users/yugao/UOP/ORS-processing/data/processed/stratus/{case_name}_sbe16_10600.nc'
# Load the dataset
# Replace 'your_data.nc' with the path to your dataset file
ds = xr.open_dataset(netcdf_path)
ds2 = xr.open_dataset(netcdf_path2)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/yugao/UOP/ORS-processing/data/processed/stratus/stratus14_sbe37_10601.nc'

In [ ]:
time_coverage_start_str = ds.attrs['time_coverage_start']

In [ ]:
# Parse the start date string into a cftime object
# The exact function to use depends on your calendar; assuming Gregorian here
time_coverage_start = cftime.num2date(0, units=f"days since {time_coverage_start_str}", calendar='gregorian')
# Define an end date, for instance, one year after the start
end_date = cftime.DatetimeGregorian(time_coverage_start.year + 1, time_coverage_start.month, time_coverage_start.day)

In [ ]:
time_coverage_start_str = ds.attrs['time_coverage_start']

In [ ]:
temp = ds.temp.data

In [ ]:
truncated_ds = truncate_valid_hourly_data(ds)

In [ ]:
truncated_ds.latitude, truncated_ds.longitude

### Plot segment of the time series

In [ ]:
# Example definitions for the needed variables and parameters
hour_start = 0  # Starting hour index for the plot
hour_end = 720   # Ending hour index for the plot, e.g., 720 for the first month

In [ ]:
# Extract serial numbers from filenames
serial_number1 = netcdf_path.split('_')[-1].split('.')[0]  # Extracts '1876' from netcdf_path
serial_number2 = netcdf_path2.split('_')[-1].split('.')[0]  # Extracts '1879' from netcdf_path2


In [ ]:
# Assuming the necessary functions and variables are defined
truncated_ds = truncate_valid_hourly_data(ds)
truncated_ds2 = truncate_valid_hourly_data(ds2)

# Convert cftime to Pandas Timestamp
regular_time_array = convert_cftime_to_pandas_timestamp(truncated_ds, start_date=ds.attrs['time_coverage_start'])
regular_time_array2 = convert_cftime_to_pandas_timestamp(truncated_ds2, start_date=ds2.attrs['time_coverage_start'])

# Variables, colors, labels
variables = ['temp', 'sal', 'sal_sbe', 'cond']
colors = ['blue', 'green', 'red', 'purple']
colors2 = ['cyan', 'lightgreen', 'pink', 'violet']
labels = ['Temperature', 'Salinity', 'Salinity Instrument', 'Conductivity']
panel_titles = ['Temperature Profile', 'Salinity Measurements', 'Secondary Salinity Instrument', 'Conductivity Levels']

fig, axs = plt.subplots(len(variables), 1, figsize=(12, 15), sharex=True)

for i, var in enumerate(variables):
    if var in truncated_ds.variables and var in truncated_ds2.variables:
        # Plotting the first dataset
        axs[i].plot(regular_time_array[hour_start:hour_end], truncated_ds[var].values[hour_start:hour_end], 
                    label=f'{var} (SBE {serial_number1})', color=colors[i])
        # Plotting the second dataset
        axs[i].plot(regular_time_array2[hour_start:hour_end], truncated_ds2[var].values[hour_start:hour_end], 
                    label=f'{var} (SBE {serial_number2})', color=colors2[i])
        
        # Calculating correlation
        correlation = np.corrcoef(truncated_ds[var].values, truncated_ds2[var].values)[0, 1]
        axs[i].set_ylabel(labels[i])
        axs[i].set_title(panel_titles[i])  # Set title for each panel
        axs[i].legend(title=f'Correlation: {correlation:.2f}')

# Shorter anchor position format
latitude = truncated_ds.latitude if hasattr(truncated_ds, 'latitude') else 0
longitude = truncated_ds.longitude if hasattr(truncated_ds, 'longitude') else 0
formatted_latitude = f"{-1 * latitude:.2f}"  # Format to 2 decimal places
formatted_longitude = f"{-1 * longitude:.2f}"

axs[-1].set_xlabel('Time')
fig.suptitle(f'Comparison within {case_name} Data\nAnchor location: ({formatted_latitude}$^\circ$S, {formatted_longitude}$^\circ$W)', fontsize=16)

# Adjust layout and save the plot
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plot_path = '../../img/'
if not os.path.exists(plot_path):
    os.makedirs(plot_path)
plot_filename = os.path.join(plot_path, f"segment_{case_name}_{serial_number1}_vs_{serial_number2}_comparison.png")
plt.savefig(plot_filename)
plt.show()

print(f'Plot saved as {plot_filename}')


## Compare two sensors 

In [ ]:
fig, axs = plt.subplots(len(variables), 1, figsize=(12, 15), sharex=True)

for i, var in enumerate(variables):
    if var in truncated_ds.variables and var in truncated_ds2.variables:
        # Plot variable from the first dataset
        axs[i].plot(regular_time_array[:-24], truncated_ds[var].values[:-24], label=f'{var} (SBE {serial_number1})', color=colors[i])
        # Plot variable from the second dataset
        axs[i].plot(regular_time_array2[:-24], truncated_ds2[var].values[:-24], label=f'{var} (SBE {serial_number2})', color=colors2[i])
        axs[i].set_ylabel(labels[i])
        axs[i].legend()
        # Calculating correlation
        correlation = np.corrcoef(truncated_ds[var].values, truncated_ds2[var].values)[0, 1]
        axs[i].set_ylabel(labels[i])
        axs[i].set_title(panel_titles[i])  # Set title for each panel
        axs[i].legend(title=f'Correlation: {correlation:.2f}')

axs[-1].set_xlabel('Time')
fig.suptitle(f'Comparison within {case_name} Data\nAnchor location: ({formatted_latitude}$^\circ$S, {formatted_longitude}$^\circ$W)', fontsize=16)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to accommodate the title
# plt.show()

# Save the figure
plot_path = '../../img/'
if not os.path.exists(plot_path):
    os.makedirs(plot_path)  # Ensure the directory exists
plot_filename = os.path.join(plot_path, f"{case_name}_{serial_number1}_vs_{serial_number2}_comparison.png")
plt.savefig(plot_filename)
print(f'Plot saved as {plot_filename}')


In [ ]:
ds

## Scatter plot